# From automatic to manual

A reaching task supplies trial inputs and scores the state trajectories produced by a model. Start with the pre-built point-mass pairing, then assemble the same kind of loop using its controller, channels, and mechanics separately.

In [ ]:
import jax.random as jr

from feedbax.xabdeef import point_mass_nn_simple_reaches

model_key, eval_key = jr.split(jr.key(0))
context = point_mass_nn_simple_reaches(
    n_steps=8, hidden_size=12, eval_n_directions=4, key=model_key
)
task = context.task
states, losses = task.eval_with_loss(context.model, eval_key)
states.mechanics.effector.pos.shape

## Building the model ourselves using core Feedbax

The controller receives the task input and delayed position and velocity measurements. The mechanics node advances the point mass using the controller's command.

In [ ]:
from feedbax.mechanics import Mechanics
from feedbax.mechanics.plant import DirectForceInput
from feedbax.mechanics.skeleton.pointmass import PointMass
from feedbax.models.feedback import SimpleFeedback
from feedbax.models.networks import recurrent_controller
from feedbax.runtime.channel import ChannelSpec
from feedbax.runtime.feedback import StateSelector

mechanics = Mechanics(DirectForceInput(PointMass(mass=1.0)), dt=0.05)
feedback = ChannelSpec(
    where=StateSelector(("plant.skeleton.pos", "plant.skeleton.vel")),
    delay=2,
)
controller = recurrent_controller(
    SimpleFeedback.get_nn_input_size(task, mechanics, feedback_spec=feedback),
    hidden_size=12,
    out_size=2,
    key=model_key,
)
model = SimpleFeedback(
    controller, mechanics, feedback_spec=feedback, motor_noise_func=None
).prepare()
states, losses = task.eval_with_loss(model, eval_key)
states.net.hidden.shape

## Selecting part of the model to train

The graph exposes the controller at `net` and its recurrent cell and readout at `net.cell` and `net.readout`. Use the graph paths to select the parameters you need. The corresponding [native graph tutorial](/feedbax/examples/5_model_stages) shows how to export and compile a graph specification.

In [ ]:
controller = model.get_node("net")
readout = model.get_node("net.readout")
readout.layer.weight.shape